# Dataset preprocessing
Download the dataset from https://huggingface.co/datasets/PaulLerner/21-EuroParl/tree/main/data. Combine the splits into one huge dataset using `prepare_dataset_pandas`.Combine parts of the speeches with the same ID using `collapse()`.

In [ ]:
import pandas as pd

# This method is used to combine parts of the speeches with the same ID.
def collapse(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return " ".join(map(str, series))


def prepare_dataset_pandas() -> pd.DataFrame:
    splits = {
        "train": "./data/EuroParl//raw/train-00000-of-00001.parquet",
        "dev":   "data/EuroParl//raw/dev-00000-of-00001.parquet",
        "test":  "data/EuroParl/raw/test-00000-of-00001.parquet",
    }

    dfs = [
        pd.read_parquet(path)
        for path in splits.values()
    ]

    df = pd.concat(dfs, ignore_index=True)

    df_grouped = (
        df
        .groupby("speech", as_index=False)
        .agg(collapse)
    )

    df_grouped.drop(columns=["csv_index", "__index_level_0__", "split"], inplace=True)
    
    return df_grouped

Next, parse the EU parties and drop speeches from Non-Inscrits (MEPs not affiliated with any party). Finally, save to a .csv file for quicker access with `df = pd.read_csv("data/full.csv")`.

In [ ]:
df = prepare_dataset_pandas()

df["EU Party"] = df["EU Party"].apply(lambda x: x.split('/')[-1])

df = df[~df["EU Party"].str.contains(r"\bNA\b", case=False, na=False)]

df.to_csv("data/EuroParl/preprocessed/full.csv")
len(df)

In [ ]:
df = pd.read_parquet("data/EuroParl/raw/train-00000-of-00001.parquet").groupby("speech", as_index=False).agg(collapse).drop(columns=["csv_index", "__index_level_0__", "split"], inplace=True)
df["EU Party"] = df["EU Party"].apply(lambda x: x.split('/')[-1])

df = df[~df["EU Party"].str.contains(r"\bNA\b", case=False, na=False)]

df.to_csv("data/EuroParl/preprocessed/2009filtered.csv")